# Vidya — Real M2 Screw Dataset Download + MobileNetV3 Training

**Dataset:** M2 screw defect detection, Mendeley Data V2, 5,536 images, CC BY 4.0 as stated by publisher. BODY.rar (2.72 GB) + HEAD.rar (1.02 GB).

**Important:** this trains two narrow OK/NG defect classifiers (Body and Head), not a general machine-part recognizer. It cannot identify OEM fitment or provide repair steps. The original test split is kept separate. Run on a Colab runtime with several GB of free storage; GPU is recommended but not guaranteed on free tier.

Source: https://data.mendeley.com/datasets/2y6mcsyrpy/2

## 1. Install tools and prepare workspace
Run cells from top to bottom. Keep this tab open during the multi-GB download.

In [ ]:
!apt-get -qq update && apt-get -qq install -y p7zip-full
!pip -q install requests tqdm pillow torch torchvision
from pathlib import Path
WORK = Path('/content/vidya_m2')
RAW = WORK / 'archives'
EXTRACT = WORK / 'extracted'
OUT = WORK / 'models'
for p in (RAW, EXTRACT, OUT): p.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORK)


## 2. Fetch official Mendeley file metadata
This asks the public dataset API for version 2 and prints the file names, sizes and available checksums before downloading. If the API requires authorization or returns a different response, stop and use the publisher's Download All button on the dataset page rather than guessing URLs.

In [ ]:
import requests, json, re, os, hashlib, time
from urllib.parse import urljoin
DATASET_ID = '2y6mcsyrpy'
API = 'https://api.data.mendeley.com'
headers = {'Accept':'application/json, application/vnd.mendeley-public-dataset.1+json'}
meta_resp = requests.get(f'{API}/datasets/{DATASET_ID}?version=2', headers=headers, timeout=60)
print('Dataset metadata HTTP:', meta_resp.status_code)
meta_resp.raise_for_status()
metadata = meta_resp.json()
print('Dataset:', metadata.get('name'), '| version:', metadata.get('version'))
files_resp = requests.get(f'{API}/datasets/{DATASET_ID}/files?version=2&$limit=100', headers=headers, timeout=60)
print('Files HTTP:', files_resp.status_code)
files_resp.raise_for_status()
file_payload = files_resp.json()
if isinstance(file_payload, dict):
    file_items = file_payload.get('files') or file_payload.get('results') or file_payload.get('items') or []
else:
    file_items = file_payload
print('File records returned:', len(file_items))
for item in file_items:
    details = item.get('content_details') or {}
    print(item.get('filename') or item.get('name'), '| bytes:', item.get('size') or details.get('size'), '| id:', item.get('id'))
assert file_items, 'No file records returned. Stop; do not guess download URLs.'
targets = [x for x in file_items if (x.get('filename') or x.get('name') or '').lower().endswith(('.rar','.zip'))]
assert targets, 'No archive file names found in API response.'
print('Candidate archives:', [x.get('filename') or x.get('name') for x in targets])


## 3. Download the original archives
Downloads stream to disk and can take time. The code checks the final file size against the API metadata when available. It does not invent a checksum if the publisher API does not supply one.

In [ ]:
from tqdm.auto import tqdm
downloaded = []
for item in targets:
    name = Path(item.get('filename') or item.get('name')).name
    file_id = item.get('id')
    assert file_id, f'Missing file id for {name}; stop rather than guessing.'
    dest = RAW / name
    if not dest.exists():
        endpoint = f'{API}/datasets/{DATASET_ID}/files/{file_id}/file_downloaded?version=2'
        with requests.get(endpoint, headers=headers, stream=True, allow_redirects=True, timeout=(30,180)) as resp:
            print(name, 'download HTTP:', resp.status_code, 'final URL host:', requests.utils.urlparse(resp.url).netloc)
            resp.raise_for_status()
            total = int(resp.headers.get('content-length') or 0)
            with open(dest, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, desc=name) as bar:
                for chunk in resp.iter_content(chunk_size=8*1024*1024):
                    if chunk:
                        f.write(chunk); bar.update(len(chunk))
    expected = item.get('size') or (item.get('content_details') or {}).get('size')
    actual = dest.stat().st_size
    print(name, 'downloaded bytes:', actual, '| API size:', expected)
    if expected is not None: assert actual == int(expected), f'Size mismatch for {name}'
    hasher = hashlib.sha256()
    with open(dest, 'rb') as archive_stream:
        for chunk in iter(lambda: archive_stream.read(8 * 1024 * 1024), b''):
            hasher.update(chunk)
    digest = hasher.hexdigest()
    expected_sha = (item.get('content_details') or {}).get('sha256_hash')
    if expected_sha:
        assert digest.lower() == expected_sha.lower(), f'SHA256 mismatch for {name}'
        print('Publisher SHA256 verified:', name, digest)
    else:
        print('SHA256 computed (publisher checksum unavailable):', name, digest)
    downloaded.append(dest)
print('Archives ready:', [p.name for p in downloaded])


## 4. Extract and inspect the archive layout
Only regular files from the publisher's RAR archives are extracted. The next cell searches for Body/Head train/test class folders.

In [ ]:
import subprocess, re
from pathlib import PurePosixPath, Path

def preflight_7z(archive_path):
    listing = subprocess.run(['7z','l','-slt',str(archive_path)], text=True, capture_output=True, check=True)
    records = []
    current = {}
    for line in listing.stdout.splitlines():
        if not line.strip():
            if current: records.append(current); current = {}
            continue
        if ' = ' in line:
            key, value = line.split(' = ', 1)
            current[key] = value
    if current: records.append(current)
    checked = 0
    for record in records:
        name = record.get('Path')
        if not name or name == str(archive_path): continue
        normalized = name.replace('\\\\', '/')
        path = PurePosixPath(normalized)
        if path.is_absolute() or re.match(r'^[A-Za-z]:', normalized) or '..' in path.parts:
            raise RuntimeError(f'Unsafe archive path rejected: {name!r}')
        if record.get('Symbolic Link') or record.get('Hard Link'):
            raise RuntimeError(f'Archive link rejected: {name!r}')
        checked += 1
    print(f'Archive preflight passed: {archive_path.name} | entries checked: {checked}')

for archive in downloaded:
    preflight_7z(archive)
    destination = EXTRACT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(['7z','x','-y',f'-o{destination}',str(archive)], text=True, capture_output=True)
    print(result.stdout[-2500:])
    if result.returncode not in (0,1):
        raise RuntimeError(f'7z extraction failed for {archive.name}: {result.stderr[-2000:]}')
    root_resolved = destination.resolve()
    for extracted in destination.rglob('*'):
        if not extracted.resolve().is_relative_to(root_resolved):
            raise RuntimeError(f'Extracted path escaped destination: {extracted}')
print('Top-level extracted paths:', [str(p.relative_to(EXTRACT)) for p in list(EXTRACT.iterdir())[:30]])
image_exts = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
for part in ('Body','Head'):
    candidates = [p for p in EXTRACT.rglob(part) if p.is_dir()]
    print(part, 'candidate directories:', [str(x) for x in candidates[:5]])
    for folder in candidates[:1]:
        for split in ('Train','Test'):
            for cls in ('OK','NG'):
                d = folder / split / cls
                if d.exists():
                    count = sum(1 for f in d.rglob('*') if f.suffix.lower() in image_exts)
                    print(part, split, cls, count)


## 5. Prepare a validation split without touching the publisher test set
A 15% validation subset is sampled from the publisher's Train folder. The original Test folder is copied untouched. The script stops if the expected folders are not found.

In [ ]:
import shutil, random
random.seed(42)
def find_part_dir(part):
    matches = [p for p in EXTRACT.rglob(part) if p.is_dir() and all((p/s/c).is_dir() for s in ('Train','Test') for c in ('OK','NG'))]
    if not matches: raise FileNotFoundError(f'Could not find {part}/Train|Test/OK|NG layout. Inspect extracted paths and stop.')
    return matches[0]
def copy_images(files, target):
    target.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(files):
        shutil.copy2(src, target / f'{i:06d}_{src.name}')
prepared = {}
for part in ('Body','Head'):
    source = find_part_dir(part)
    root = WORK / f'{part.lower()}_imagefolder'
    if root.exists(): shutil.rmtree(root)
    for cls in ('OK','NG'):
        train_files = sorted([p for p in (source/'Train'/cls).rglob('*') if p.suffix.lower() in image_exts])
        test_files = sorted([p for p in (source/'Test'/cls).rglob('*') if p.suffix.lower() in image_exts])
        random.shuffle(train_files)
        n_val = max(1, round(len(train_files)*0.15))
        val_files, fit_files = train_files[:n_val], train_files[n_val:]
        copy_images(fit_files, root/'train'/cls)
        copy_images(val_files, root/'val'/cls)
        copy_images(test_files, root/'test'/cls)
    prepared[part] = root
    print(part, {s:{c:len(list((root/s/c).glob('*'))) for c in ('OK','NG')} for s in ('train','val','test')})


## 6. Train two separate MobileNetV3-Small baselines
This downloads the existing Vidya trainer from the repository and trains Body and Head separately. Pretrained ImageNet weights require internet access. A free Colab GPU may not always be available.

In [ ]:
TRAINER_URL = 'https://raw.githubusercontent.com/anupamsarma997-debug/Vidya-AI-System./vidya-foundation/tools/train_imagefolder_baseline.py'
trainer_path = WORK / 'train_imagefolder_baseline.py'
r = requests.get(TRAINER_URL, timeout=60); r.raise_for_status(); trainer_path.write_bytes(r.content)
print('Trainer downloaded:', trainer_path, '| bytes:', trainer_path.stat().st_size)
for part, data_root in prepared.items():
    output = OUT / part.lower()
    cmd = ['python', str(trainer_path), '--data-root', str(data_root), '--output-dir', str(output), '--epochs', '10', '--batch-size', '32']
    print('Starting', part, 'training:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    print('Completed', part, '| output:', output)


## 7. Review results and download model artifacts
Do not interpret this as general Vidya machine recognition. Review test metrics, per-class performance, sample errors and dataset license attribution before using any checkpoint.

In [ ]:
for part in ('body','head'):
    metrics_path = OUT / part / 'test_metrics.json'
    print('\n', part.upper())
    print(metrics_path.read_text() if metrics_path.exists() else 'No metrics file: training did not finish')
from google.colab import files
import shutil
bundle = shutil.make_archive('/content/vidya_m2_model_artifacts', 'zip', OUT)
print('Artifact bundle:', bundle)
files.download(bundle)


## Dataset attribution
Rizki Putra Prastio and Rodik Wahyu Indrawan (2026), “M2 screw defect detection dataset”, Mendeley Data, Version 2, DOI: 10.17632/2y6mcsyrpy.2. License stated on dataset page: Creative Commons Attribution 4.0 International (CC BY 4.0). Keep this citation and license notice with any permitted reuse.

**Limitations:** results only describe the M2 screw body/head OK-vs-NG dataset. They do not identify screw sizes, OEM part numbers, machine compatibility, other components, causes of failure, or safe repair actions.